# 05 · Semáforo, personas en juego y archivos para el mapa

> **Reto 2 del INE:** "¿Qué zonas podrían presentar mayor abstención en el futuro si no se interviene?"

**A qué responde este paso.** El notebook 04 dejó una desviación prevista para cada municipio en la intermedia del 6 de junio de 2027. Aquí la convertimos en algo que el INE pueda usar:

1. **Un semáforo** que diga dónde está el mayor riesgo, con **qué tan seguro** es cada color.
2. **Las personas en juego** en cada municipio: a quién dirigir la intervención, contado con datos y sin suponer cuánto funcionaría una campaña.
3. **El contexto de violencia y seguridad**, como capa aparte del riesgo.
4. **Los tres archivos que lee el mapa** (`municipios.csv`, `escenarios_2027.csv` y `metadatos.json`), con el formato de `docs/contrato-datos.md`.

**Decisiones de Sofía Ordaz del 20 de octubre de 2026** (detalle en `docs/bitacora.md`):

| # | Decisión | Qué se adoptó |
|---|---|---|
| **D17** | Dato de violencia para el mapa | Homicidios del INEGI por municipio, con una etiqueta por estado de qué fuente cuenta más (notebook 03b, sección 8) |
| **D18** | Cortes del semáforo | **Rojo** = 20 % de municipios de mayor riesgo; **amarillo** = el 20 % siguiente; **verde** = el resto |
| **D22** | Escenario con intervención | Sí, como **cuenta con tres supuestos explícitos** en los municipios en rojo, con la etiqueta "supuesto, no efecto medido" (sección 5b). No cambia D19 |
| **D27** (22 de octubre) | Una sola predicción | El nivel nacional de 2027 se estima con un modelo por calendario (sección 0): **51.4 %**. Se muestra una sola predicción por municipio; los niveles de 2009, 2015 y 2021 quedan como sensibilidad en la documentación |
| **D28** (22 de octubre) | Tipos de intervención | Cada supuesto por separado (retener primerizos, jóvenes de 20 a 29 años, credencialización) y los tres juntos |
| **D29** (22 de octubre) | Por qué de cada predicción | Se agrega a `municipios.csv` cuánto aporta cada grupo de variables a la predicción, frente al país (notebook 04c) |
| **D31** (24 de octubre) | Zaragoza 2012 | Con lista incompleta se usan todas las actas aunque haya mucho "no sabemos" (notebook 03) |
| **D32** (24 de octubre) | Intervención con componentes sin dato | Se suma solo lo conocido; si falta S3, tampoco se agregan personas a la lista, y el escenario se marca `intervencion_parcial` |
| **D33** (24 de octubre) | Modelo final | Delta con guardia: parte de la presidencial anterior y aprende el cambio (notebook 04) |
| **D19** | Escenario "con campaña dirigida" | **No** se presenta como "la participación subiría X puntos", porque no tenemos ninguna medición de cuánto funciona una campaña. Se muestran las **personas en juego** en cada municipio |

## Resumen de lo que encontramos

1. **Rojo: 495 municipios con 25.0 millones de personas en la lista de 2024.** De ellos, **307 son "rojos firmes"** (probabilidad de rojo de 70 % o más, 13.2 millones de personas) y 188 están en el límite.
2. **El rojo tiene dos historias distintas, que piden intervenciones distintas:**
   - **404 municipios sin elección municipal ese día** (6.2 millones de personas): 299 de usos y costumbres en Oaxaca, 69 de Veracruz, 29 de Durango y 7 que eligen por sistemas normativos indígenas fuera de Oaxaca. Ahí la intermedia solo trae la elección federal (y, en algunos, diputaciones locales).
   - **91 municipios que sí eligen ayuntamiento** (18.8 millones de personas), pero que votan poco siempre: Tijuana, Juárez, Mexicali, Culiacán, Hermosillo, Reynosa y otras ciudades, sobre todo del norte.
3. **Personas en juego en los municipios en rojo:**
   - 619 mil primerizos que votaron en 2024, la generación que más cae en la intermedia;
   - 3.0 millones de jóvenes de 21 a 29 años que no votaron;
   - 1.2 millones de nuevos votantes en 2027, de los que unos 332 mil quedarían sin credencial si se repite la credencialización de 2021.
4. **Una sola predicción por municipio (D27).** El nivel nacional de 2027 se estima en **51.4 %** con el calendario local: el 91 % de la lista elige ayuntamiento ese día. Probado con las tres intermedias, su error es de unos 3 puntos. **El nivel exacto de cada municipio es incierto; su posición, no tanto.** El rango de 90 % suma el error del municipio (de unos ±7 a ±15 pp según su tamaño) y el del nivel nacional: mide de 18 a 29 puntos de ancho. Es una aproximación: probado de una elección a otra, cubrió 79 % y 93 %. La probabilidad de rojo es razonable al predecir 2021 y optimista al predecir 2015: cuando dice 80 % o más, conviene leerla como "probable", no como "seguro".
5. **La violencia casi no pesa:** la tasa de homicidios es una de las 34 variables del modelo, pero su aporte es casi nulo, y los municipios en rojo no son más violentos que los demás. El 54 % de ellos no tiene policía municipal, por los municipios de Oaxaca con policía comunitaria.
6. **Con intervención (D22, D28), según el tipo:** retener a los primerizos saca del rojo a 19 municipios; cerrar la mitad de la brecha joven, a 68; credencializar, a 7; y los tres juntos, a 84 (411 siguen en rojo). Juntos suman 528 mil votos: +1.59 pp en el rojo típico, que está a 5.7 pp de la línea. Las campañas para jóvenes ayudan, pero no compensan la falta de elección municipal.
7. **Los tres archivos del mapa** quedan en `data/processed/mapa/resultados-1.6/`, con el formato del contrato de datos. La 1.6 corrige Zaragoza 2012 (D31), marca las intervenciones con componentes sin dato (D32) y usa el modelo Delta con guardia (D33).

## Palabras que usamos

| Palabra | Qué significa |
|---|---|
| **Posición de riesgo** | De 0 a 100. **100 = el municipio que se prevé más por debajo del país.** Es `100 − percentil_riesgo` del notebook 04, donde el sentido era el contrario. |
| **Semáforo** | Rojo, amarillo o verde según la posición de riesgo (D18). |
| **Probabilidad de rojo** | En cuántas de 2,000 simulaciones el municipio queda en rojo cuando a su predicción se le suma un error del tamaño de los que el modelo tuvo en la validación. |
| **Intervalo de 90 %** | Rango donde esperamos la participación del municipio. Suma los errores reales de la validación del municipio y el error del nivel nacional estimado. |
| **Personas en juego** | Grupos concretos de la lista nominal donde una intervención tendría a quién llegar. Son conteos, no efectos. |

In [1]:
from datetime import datetime
from pathlib import Path
import json
import warnings

import numpy as np
import polars as pl
from scipy.stats import rankdata

warnings.filterwarnings('ignore')
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(14)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_width_chars(200)

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() and (p / 'data').is_dir())
P, E = raiz / 'data/processed', raiz / 'data/processed/externos'
requeridos = ['riesgo_2027.parquet', 'aportes_2027.parquet', 'validacion_modelo.parquet', 'indicadores_mapa.parquet', 'panel_municipal.parquet', 'secciones_territorio_2024.parquet',
              'lista_edad_sexo_municipal.parquet', 'municipios_para_mapa.csv']
faltan = [f for f in requeridos if not (P / f).exists()]
assert not faltan, f'Primero corre los notebooks 03, 03b, 04 y 04c; faltan {faltan}'

riesgo = pl.read_parquet(P / 'riesgo_2027.parquet')
aportes = pl.read_parquet(P / 'aportes_2027.parquet')
validacion = pl.read_parquet(P / 'validacion_modelo.parquet')
indicadores = pl.read_parquet(P / 'indicadores_mapa.parquet')
panel = pl.read_parquet(P / 'panel_municipal.parquet')
variables = pl.read_parquet(P / 'variables_transiciones.parquet')
rng = np.random.default_rng(2027)
print(f'Municipios con riesgo 2027: {riesgo.height:,} · sin predicción: {riesgo["desviacion_prevista"].null_count()}')

Municipios con riesgo 2027: 2,477 · sin predicción: 0


## 0 · ¿Cuánto votará el país? Un modelo nacional por calendario (D27)

El modelo del notebook 04 predice la **posición** de cada municipio frente al país, no el nivel nacional. Para dar **una sola predicción** de participación por municipio hace falta estimar ese nivel.

**Qué mueve el nivel nacional de una intermedia:** el calendario local. Cuantas más personas eligen ayuntamiento el mismo día, más vota el país (notebook 03b). Ajustamos una recta con las tres intermedias de los CCPC: participación nacional según el % de la lista que elige ayuntamiento ese día.

**Cómo lo probamos:** predecimos cada intermedia con las otras dos. **Advertencia:** son solo tres elecciones; el resultado es una estimación razonable, no una predicción precisa.

In [2]:
NAC = {2009: 44.06, 2015: 47.07, 2021: 51.84}   # participación nacional de las intermedias según los CCPC (notebook 01)
calendario = (panel.filter(pl.col('AELEC').is_in(list(NAC))).group_by('AELEC').agg(
                  pct_lista_ayuntamiento=(pl.col('LN') * pl.col('ayuntamiento_mismo_dia').cast(pl.Float64)).sum() / pl.col('LN').sum() * 100)
              .sort('AELEC').with_columns(participacion_nacional=pl.col('AELEC').replace_strict(NAC, return_dtype=pl.Float64)))
f27 = variables.filter(pl.col('eleccion_objetivo') == 2027)
PCT_AYTO_2027 = (f27['lista_presidencial'] * f27['ayuntamiento_mismo_dia_intermedia']).sum() / f27['lista_presidencial'].sum() * 100
x, y = calendario['pct_lista_ayuntamiento'].to_numpy(), calendario['participacion_nacional'].to_numpy()
filas = []
for i, anio in enumerate(calendario['AELEC'].to_list()):
    otras = np.arange(len(x)) != i
    b1, b0 = np.polyfit(x[otras], y[otras], 1)
    filas.append({'Intermedia': anio, '% de la lista con ayuntamiento ese día': round(x[i], 1), 'Participación real (%)': y[i],
                  'Predicha con las otras dos (%)': round(b0 + b1 * x[i], 1), 'Error (pp)': round(b0 + b1 * x[i] - y[i], 1)})
prueba_nacional = pl.DataFrame(filas)
PENDIENTE, ORDENADA = np.polyfit(x, y, 1)
NIVEL_2027 = round(float(ORDENADA + PENDIENTE * PCT_AYTO_2027), 2)
ERROR_NACIONAL = float(np.sqrt(np.mean(prueba_nacional['Error (pp)'].to_numpy() ** 2)))   # raíz del error cuadrático medio de la prueba
ESCENARIOS = {'estimado': NIVEL_2027}
SENSIBILIDAD = {'como 2009': 44.06, 'como 2015': 47.07, 'como 2021': 51.84}
print(f'Recta: participación nacional = {ORDENADA:.2f} + {PENDIENTE:.3f} × % de la lista con ayuntamiento')
print(f'2027: {PCT_AYTO_2027:.1f} % de la lista elige ayuntamiento ese día → nivel nacional estimado {NIVEL_2027:.1f} %')
print(f'Error de la prueba (raíz del error cuadrático medio): {ERROR_NACIONAL:.1f} pp')

def participacion_municipal(nivel):
    # Mismo cálculo que el notebook 04: nivel + desviación prevista + un ajuste común para que el promedio ponderado por la lista sea el nivel
    pesos = riesgo['lista_2024'].fill_null(0).to_numpy().astype(float)
    bruta = nivel + riesgo['desviacion_prevista'].to_numpy()
    return np.clip(bruta + nivel - np.average(bruta, weights=pesos), 0, 100)
assert np.allclose(participacion_municipal(47.07), riesgo['participacion_prevista_medio'].to_numpy()), 'no reproduce el cálculo del notebook 04'
riesgo = riesgo.with_columns(participacion_prevista_estimado=pl.Series(participacion_municipal(NIVEL_2027)))
prueba_nacional

Recta: participación nacional = 37.43 + 0.153 × % de la lista con ayuntamiento
2027: 91.2 % de la lista elige ayuntamiento ese día → nivel nacional estimado 51.4 %
Error de la prueba (raíz del error cuadrático medio): 3.3 pp


Intermedia,% de la lista con ayuntamiento ese día,Participación real (%),Predicha con las otras dos (%),Error (pp)
i64,f64,f64,f64,f64
2009,47.0,44.06,45.7,1.6
2015,57.9,47.07,45.8,-1.2
2021,95.1,51.84,57.3,5.4


**Qué vemos:**

- **La recta reproduce bien 2009 y 2015** (errores de +1.6 y −1.2 puntos) y **sobreestima 2021 por 5.4 puntos**: con casi toda la lista votando su ayuntamiento, la participación subió menos de lo que la recta esperaba. Su error típico es de 3.3 puntos.
- **Para 2027 da 51.4 %,** casi el nivel de 2021 (51.8 %), que es la intermedia con el calendario más parecido.
- **Es una estimación con solo tres elecciones:** el rango de cada municipio incluye este error. **El semáforo no depende del nivel nacional**, solo del orden de los municipios; con otro nivel, todos los porcentajes se mueven casi lo mismo.

## 1 · ¿Cuánto se puede equivocar el modelo en un municipio?

El intervalo del notebook 04 (bootstrap) solo mide cuánto cambia el modelo si se entrena con otros municipios: unos 1.7 pp. **No incluye el error de una elección nueva**, que es mucho mayor. Para el mapa usamos ese segundo error, medido en la validación:

- **Errores de la validación:** real menos previsto en las dos pruebas del notebook 04 (aprender de 2015 y predecir 2021, y al revés). Les quitamos el desplazamiento general de cada elección, porque el error del nivel nacional se suma aparte (D27).
- **Por tamaño del municipio:** los municipios chicos varían más. Separamos los errores en cuatro grupos según su lista nominal.

**Qué esperamos:** intervalos más anchos en municipios chicos, y que un intervalo construido con una elección cubra cerca del 90 % de la otra.

In [3]:
def errores_centrados(d):
    e = (d['real'] - d['prevista']).to_numpy()
    return d.with_columns(error=pl.Series(e - np.average(e, weights=d['lista_presidencial'].to_numpy().astype(float))))

TAMANOS = [5_000, 20_000, 100_000]
ETIQUETAS_TAMANO = ['menos de 5 mil', '5 a 20 mil', '20 a 100 mil', '100 mil o más']
def grupo_tamano(lista):
    return pl.Series(np.digitize(lista, TAMANOS))

errores = pl.concat([errores_centrados(validacion.filter(pl.col('validacion') == v)) for v in validacion['validacion'].unique().sort()])
errores = errores.with_columns(grupo=grupo_tamano(errores['lista_presidencial'].to_numpy()))

# ¿Cubre? Intervalo armado con una validación, probado en la otra
filas = []
for armado, probado in [('aprende de 2015, predice 2021', 'aprende de 2021, predice 2015'), ('aprende de 2021, predice 2015', 'aprende de 2015, predice 2021')]:
    q = errores.filter(pl.col('validacion') == armado).group_by('grupo').agg(q05=pl.col('error').quantile(0.05), q95=pl.col('error').quantile(0.95))
    dentro = errores.filter(pl.col('validacion') == probado).join(q, on='grupo').select(((pl.col('error') >= pl.col('q05')) & (pl.col('error') <= pl.col('q95'))).mean()).item()
    filas.append({'Intervalo armado con': armado, 'Probado en': probado, 'Cobertura real (meta: 90 %)': round(dentro * 100, 1)})
print(pl.DataFrame(filas))

# Para 2027: errores de las dos validaciones juntos (lo más prudente)
cuantiles = (errores.group_by('grupo').agg(pl.len().alias('Errores'), q05=pl.col('error').quantile(0.05), q95=pl.col('error').quantile(0.95),
                                            mediana_abs=pl.col('error').abs().median()).sort('grupo'))
cuantiles.with_columns(pl.col('grupo').replace_strict(dict(enumerate(ETIQUETAS_TAMANO))).alias('Lista nominal'),
                       pl.col('q05', 'q95', 'mediana_abs').round(1)).select(
    'Lista nominal', 'Errores', pl.col('q05').alias('Percentil 5 (pp)'), pl.col('q95').alias('Percentil 95 (pp)'), pl.col('mediana_abs').alias('Error típico (pp)'))

shape: (2, 3)
┌───────────────────────────────┬───────────────────────────────┬─────────────────────────────┐
│ Intervalo armado con          ┆ Probado en                    ┆ Cobertura real (meta: 90 %) │
│ ---                           ┆ ---                           ┆ ---                         │
│ str                           ┆ str                           ┆ f64                         │
╞═══════════════════════════════╪═══════════════════════════════╪═════════════════════════════╡
│ aprende de 2015, predice 2021 ┆ aprende de 2021, predice 2015 ┆ 78.8                        │
│ aprende de 2021, predice 2015 ┆ aprende de 2015, predice 2021 ┆ 93.4                        │
└───────────────────────────────┴───────────────────────────────┴─────────────────────────────┘


Lista nominal,Errores,Percentil 5 (pp),Percentil 95 (pp),Error típico (pp)
str,u32,f64,f64,f64
"""menos de 5 mil""",1698,-15.1,12.1,5.1
"""5 a 20 mil""",1791,-12.0,10.9,4.8
"""20 a 100 mil""",1058,-8.9,9.2,3.4
"""100 mil o más""",308,-7.4,9.3,3.0


**Qué vemos:**

- **Un intervalo armado con una elección cubre 93 % de la otra en un sentido y 79 % en el otro.** Con los errores de 2021 se queda corto en 2015, la intermedia del boicot magisterial. Por eso para 2027 usamos los errores de **las dos** validaciones juntas. **Es una aproximación, no una garantía:** la cobertura varía entre elecciones y entre grupos de municipios (es menor en los de usos y costumbres).
- **El intervalo es ancho:** de −15 a +12 pp en los municipios de menos de 5 mil electores, y de −7 a +9 pp en los de 100 mil o más. **El nivel exacto de participación de un municipio es incierto; su posición frente a los demás es lo confiable**, y eso es lo que usa el semáforo.

## 2 · El semáforo y qué tan seguro es cada color

- **Posición de riesgo** = 100 − percentil del notebook 04 (100 = mayor riesgo).
- **Rojo:** posición de 80 o más (20 % de mayor riesgo). **Amarillo:** de 60 a menos de 80. **Verde:** el resto (D18).
- **Probabilidad de rojo:** 2,000 simulaciones. En cada una, a la predicción de cada municipio se le suma un error tomado al azar de los errores de su mismo tamaño, y se vuelve a ordenar.

**Antes de usar la probabilidad la comprobamos** en la validación: estimamos la de 2021 con los errores de 2015, y la de 2015 con los de 2021, y vemos si los municipios con 80 % de probabilidad de rojo realmente quedaron en rojo cerca del 80 % de las veces.

In [4]:
def probabilidad_rojo(prevista, grupos, pool_error, pool_grupo, veces=2000, fraccion=0.2):
    n = len(prevista); k = int(round(n * fraccion)); cuenta = np.zeros(n)
    pools = {g: pool_error[pool_grupo == g] for g in np.unique(pool_grupo)}
    indices = {g: np.flatnonzero(grupos == g) for g in pools}
    for _ in range(veces):
        sim = prevista.copy()
        for g, pool in pools.items():
            sim[indices[g]] += rng.choice(pool, len(indices[g]))
        cuenta[np.argsort(sim)[:k]] += 1
    return cuenta / veces

filas = []
for objetivo, fuente in [('aprende de 2015, predice 2021', 'aprende de 2021, predice 2015'), ('aprende de 2021, predice 2015', 'aprende de 2015, predice 2021')]:
    o, f = errores.filter(pl.col('validacion') == objetivo), errores.filter(pl.col('validacion') == fuente)
    prob = probabilidad_rojo(o['prevista'].to_numpy(), o['grupo'].to_numpy(), f['error'].to_numpy(), f['grupo'].to_numpy(), veces=1000)
    real = o['real'].to_numpy(); k = int(round(len(real) * 0.2))
    rojo_real = np.zeros(len(real), bool); rojo_real[np.argsort(real)[:k]] = True
    tabla = (pl.DataFrame({'p': prob, 'r': rojo_real})
             .with_columns(rango=pl.when(pl.col('p') < 0.1).then(pl.lit('a. menos de 10 %')).when(pl.col('p') < 0.3).then(pl.lit('b. 10 a 30 %'))
                           .when(pl.col('p') < 0.5).then(pl.lit('c. 30 a 50 %')).when(pl.col('p') < 0.7).then(pl.lit('d. 50 a 70 %'))
                           .when(pl.col('p') < 0.9).then(pl.lit('e. 70 a 90 %')).otherwise(pl.lit('f. 90 % o más')))
             .group_by('rango').agg(pl.len().alias('Municipios'), (pl.col('p').mean() * 100).round(1).alias('Probabilidad prevista (%)'),
                                    (pl.col('r').mean() * 100).round(1).alias('Quedaron en rojo de verdad (%)'))
             .with_columns(pl.lit(objetivo.split(', ')[1]).alias('Elección probada')).sort('rango'))
    filas.append(tabla)
pl.concat(filas).select('Elección probada', pl.col('rango').alias('Probabilidad de rojo'), 'Municipios', 'Probabilidad prevista (%)', 'Quedaron en rojo de verdad (%)')

Elección probada,Probabilidad de rojo,Municipios,Probabilidad prevista (%),Quedaron en rojo de verdad (%)
str,str,u32,f64,f64
"""predice 2021""","""a. menos de 10 %""",1447,2.0,1.0
"""predice 2021""","""b. 10 a 30 %""",394,19.2,18.3
"""predice 2021""","""c. 30 a 50 %""",204,39.7,45.1
"""predice 2021""","""d. 50 a 70 %""",141,59.3,68.1
"""predice 2021""","""e. 70 a 90 %""",146,80.2,77.4
"""predice 2021""","""f. 90 % o más""",105,95.9,94.3
"""predice 2015""","""a. menos de 10 %""",1433,2.3,4.6
"""predice 2015""","""b. 10 a 30 %""",403,18.1,21.8
"""predice 2015""","""c. 30 a 50 %""",179,39.5,41.3


**Qué vemos:**

- **Al predecir 2021, la probabilidad es razonable:** cuando dice 59 %, quedaron en rojo el 68 %; cuando dice 80 %, el 77 %; cuando dice 96 %, el 94 %.
- **Al predecir 2015 (la intermedia del boicot magisterial), se pasa de segura arriba:** cuando dice 96 %, fue 80 %; cuando dice 80 %, fue 59 %.
- Por eso, para 2027 simulamos con los errores de las dos validaciones juntas, que dan probabilidades más prudentes. Aun así, **una probabilidad de 80 % o más se lee como "probable", no como "seguro"**. Es una frecuencia de simulación con los errores del pasado, no una probabilidad garantizada.

In [5]:
r = riesgo.filter(pl.col('desviacion_prevista').is_not_null())
grupos_2027 = grupo_tamano(r['lista_2024'].fill_null(0).to_numpy()).to_numpy()
prob_2027 = probabilidad_rojo(r['desviacion_prevista'].to_numpy(), grupos_2027, errores['error'].to_numpy(), errores['grupo'].to_numpy())
q = cuantiles.select('grupo', 'q05', 'q95')

semaforo = (r.select('cvegeo', 'desviacion_prevista', 'lista_2024', 'percentil_riesgo')
            .with_columns(grupo=pl.Series(grupos_2027), probabilidad_rojo=pl.Series(prob_2027),
                          posicion_riesgo=(100 - pl.col('percentil_riesgo')))
            .join(q, on='grupo', how='left')
            .with_columns(semaforo=pl.when(pl.col('posicion_riesgo') >= 80).then(pl.lit('rojo'))
                          .when(pl.col('posicion_riesgo') >= 60).then(pl.lit('amarillo')).otherwise(pl.lit('verde'))))
UMBRAL_ROJO = semaforo.filter(pl.col('semaforo') == 'rojo')['desviacion_prevista'].max()
UMBRAL_AMARILLO = semaforo.filter(pl.col('semaforo') == 'amarillo')['desviacion_prevista'].max()
print(f'Corte rojo: desviación prevista ≤ {UMBRAL_ROJO:.2f} pp · corte amarillo: ≤ {UMBRAL_AMARILLO:.2f} pp')

(semaforo.with_columns(firmeza=pl.when(pl.col('probabilidad_rojo') >= 0.7).then(pl.lit('rojo firme (70 % o más)'))
                       .when(pl.col('probabilidad_rojo') >= 0.3).then(pl.lit('en el límite (30 a 70 %)')).otherwise(pl.lit('poco probable (menos de 30 %)')))
 .group_by('semaforo', 'firmeza').agg(pl.len().alias('Municipios'), pl.col('lista_2024').sum().alias('Personas en la lista 2024'))
 .sort('semaforo', 'firmeza').rename({'semaforo': 'Semáforo', 'firmeza': 'Probabilidad de rojo'}))

Corte rojo: desviación prevista ≤ -6.82 pp · corte amarillo: ≤ 1.93 pp


Semáforo,Probabilidad de rojo,Municipios,Personas en la lista 2024
str,str,u32,i64
"""amarillo""","""en el límite (30 a 70 %)""",103,3033382
"""amarillo""","""poco probable (menos de 30 %)""",393,38285578
"""rojo""","""en el límite (30 a 70 %)""",188,11799346
"""rojo""","""rojo firme (70 % o más)""",307,13155922
"""verde""","""poco probable (menos de 30 %)""",1486,32056120


**Qué vemos:**

- El semáforo tiene 20 % de municipios en rojo (495) y 20 % en amarillo (496), por construcción.
- **No todos los rojos son igual de seguros.** 307 son "rojos firmes" y 188 están en el límite. Para priorizar recursos escasos conviene empezar por los firmes; los del límite son los primeros a vigilar con información nueva (por ejemplo, el calendario local definitivo de 2027).
- 103 municipios amarillos tienen entre 30 % y 70 % de probabilidad de rojo: están justo en el corte.

## 3 · Personas en juego

Contamos tres grupos con los datos del INE y del INEGI. **Son conteos, no efectos:** no decimos cuánto subiría la participación (D19).

1. **Primerizos de 2024:** personas de 18 a 20 años que **votaron** en 2024, su primera elección federal. En la intermedia siguiente, la generación primeriza de 2018 cayó 11.7 pp, 6.1 más que su municipio (notebook 03b, sección 1).
2. **Jóvenes de 21 a 29 años que no votaron en 2024:** el grupo con la participación más baja (notebook 03b, sección 1).
3. **Quienes votarán por primera vez en 2027:** tendrán 18 o 19 años el 6 de junio de 2027. Los estimamos con el censo de 2020: un cuarto de los que tenían 10 años, todos los de 11 y tres cuartos de los de 12. Además estimamos **cuántos quedarían sin credencial si se repite la credencialización de 2021** de su municipio.

Para los conteos de la lista usamos las secciones con el territorio de 2024 y las mismas reglas del notebook 03: una sección que no cuadra con su acta (D10) no cuenta como votó ni como no votó.

In [6]:
secciones_2024 = (pl.read_parquet(P / 'secciones_territorio_2024.parquet').filter(pl.col('AELEC') == 2024)
                  .select('EDOCVE', 'SECCION', 'municipio_2024', pl.col('no_cuadra_con_acta').fill_null(False)))
edad_2024 = (pl.scan_parquet(P / 'ccpc/ccpc-2024.parquet')
             .group_by(pl.col('EDOCVE').cast(pl.Int64), 'SECCION', 'EDAD').agg(pl.col('LN', 'SV', 'NV').sum()).collect()
             .join(secciones_2024.with_columns(pl.col('EDOCVE').cast(pl.Int64)), on=['EDOCVE', 'SECCION'], how='left'))
assert edad_2024['municipio_2024'].null_count() == 0, 'Hay secciones de 2024 sin municipio'
edad_2024 = edad_2024.with_columns(
    SV=pl.when(pl.col('no_cuadra_con_acta')).then(0).otherwise(pl.col('SV')),
    NV=pl.when(pl.col('no_cuadra_con_acta')).then(0).otherwise(pl.col('NV')))
personas = (edad_2024.group_by(pl.col('municipio_2024').alias('cvegeo')).agg(
    primerizos_votaron_2024=pl.col('SV').filter(pl.col('EDAD').is_between(18, 20)).sum(),
    jovenes_21_29_lista_2024=pl.col('LN').filter(pl.col('EDAD').is_between(21, 29)).sum(),
    jovenes_21_29_no_votaron_2024=pl.col('NV').filter(pl.col('EDAD').is_between(21, 29)).sum(),
    n_jov_2024=(pl.col('SV') + pl.col('NV')).filter(pl.col('EDAD').is_between(18, 34)).sum(),
    nv_jov_2024=pl.col('NV').filter(pl.col('EDAD').is_between(18, 34)).sum(),
    cohorte_18_20_2024=(pl.col('SV') + pl.col('NV')).filter(pl.col('EDAD').is_between(18, 20)).sum(),
    jovenes_20_29_2024=(pl.col('SV') + pl.col('NV')).filter(pl.col('EDAD').is_between(20, 29)).sum(),
    base_2024=(pl.col('SV') + pl.col('NV')).sum()))
# Si los conteos de 2024 no tienen a nadie con dato (la participación sale de las actas), los conteos por edad
# no son 0 sino "sin dato"
SIN_CONTEO = ['n_jov_2024', 'nv_jov_2024', 'primerizos_votaron_2024', 'jovenes_21_29_no_votaron_2024', 'cohorte_18_20_2024', 'jovenes_20_29_2024']
personas = personas.with_columns([pl.when(pl.col('base_2024') > 0).then(pl.col(c)).alias(c) for c in SIN_CONTEO])
print(f'Municipios sin conteo por edad en 2024 (quedan sin dato): {personas.filter(pl.col("base_2024") == 0).height}')

censo = (pl.read_parquet(E / 'inegi_edad_desplegada.parquet').filter(pl.col('anio') == 2020, pl.col('edad').is_between(10, 12))
         .group_by('cvegeo').agg(nuevos_votantes_2027=(pl.col('poblacion').filter(pl.col('edad') == 10).sum() * 0.25
                                                        + pl.col('poblacion').filter(pl.col('edad') == 11).sum()
                                                        + pl.col('poblacion').filter(pl.col('edad') == 12).sum() * 0.75).round(0).cast(pl.Int64)))
personas = (personas.join(censo, on='cvegeo', how='left')
            .join(indicadores.select('cvegeo', 'credencializacion_18_19_2021'), on='cvegeo', how='left')
            .with_columns(nuevos_sin_credencial_2027_si_se_repite_2021=(pl.col('nuevos_votantes_2027')
                          * (1 - pl.col('credencializacion_18_19_2021').clip(upper_bound=100) / 100)).round(0).cast(pl.Int64)))
print('Totales nacionales:')
personas.select(pl.col('primerizos_votaron_2024', 'jovenes_21_29_lista_2024', 'jovenes_21_29_no_votaron_2024', 'nuevos_votantes_2027',
                       'nuevos_sin_credencial_2027_si_se_repite_2021').sum())

Municipios sin conteo por edad en 2024 (quedan sin dato): 15
Totales nacionales:


primerizos_votaron_2024,jovenes_21_29_lista_2024,jovenes_21_29_no_votaron_2024,nuevos_votantes_2027,nuevos_sin_credencial_2027_si_se_repite_2021
i64,i64,i64,i64,i64
2898581,20258967,9835966,4363455,1268747


**Advertencias:**

- **Los nuevos votantes de 2027 se estiman con el censo de 2020:** no descuentan a quienes migraron o murieron desde entonces ni suman a quienes llegaron.
- **"Sin credencial si se repite 2021"** es un escenario aritmético, no una predicción. En 245 municipios no hay credencialización de 2021 (menos de 50 jóvenes en el censo), y queda vacío. Si la credencialización pasaba de 100 % (lista mayor que la población del censo), se toma como 100 %.

## 4 · Contexto de violencia (D17)

**Tasa de homicidios 2022–2024:** defunciones por homicidio del INEGI, por municipio donde ocurrieron, por cada 100 mil habitantes al año (población de CONAPO de 2023). Usamos tres años porque en municipios chicos un solo año es muy ruidoso. 2024 está un poco subestimado (2 % a 4 %) por registros tardíos.

**Aviso por estado:** qué fuente cuenta más homicidios (notebook 03b, sección 8). **Uso en el modelo:** la tasa de 2022 a 2024, en logaritmo, es una de las 34 variables (grupo de condiciones sociales) y casi no pesa; las demás columnas de seguridad son solo contexto.

In [7]:
homicidios = pl.read_parquet(E / 'inegi_homicidios_municipio.parquet').filter(pl.col('anio').is_between(2022, 2024))
poblacion_2023 = (pl.read_parquet(E / 'conapo_proyecciones_edad_sexo.parquet').filter(pl.col('anio') == 2023)
                  .group_by('cvegeo').agg(poblacion_2023=pl.col('poblacion').sum()))
violencia = (homicidios.group_by('cvegeo').agg(homicidios_2022_2024=pl.col('homicidios').sum(), anios_con_dato=pl.col('clave_presente_en_edr_anio').sum())
             .join(poblacion_2023, on='cvegeo', how='left')
             .with_columns(homicidios_2022_2024=pl.when(pl.col('anios_con_dato') == 3).then(pl.col('homicidios_2022_2024')))
             .with_columns(tasa_homicidios_2022_2024=(pl.col('homicidios_2022_2024') / 3 / pl.col('poblacion_2023') * 1e5).round(1)))
con_tasa = violencia.filter(pl.col('tasa_homicidios_2022_2024').is_not_null())
violencia = violencia.join(con_tasa.select('cvegeo', percentil_homicidios_2022_2024=pl.Series(np.round((rankdata(con_tasa['tasa_homicidios_2022_2024'].to_numpy()) - 0.5)
                                                                                                 / con_tasa.height * 100, 1))), on='cvegeo', how='left')
print(f'Municipios con tasa: {con_tasa.height:,} · sin tasa (clave ausente de la estadística algún año o sin población): {violencia.height - con_tasa.height}')

contexto = semaforo.join(violencia, on='cvegeo', how='left').join(indicadores.select('cvegeo', 'quien_ejerce_seguridad', 'comparacion_homicidios_inegi_sesnsp'), on='cvegeo', how='left')
(contexto.group_by('semaforo').agg(
    pl.len().alias('Municipios'),
    pl.col('tasa_homicidios_2022_2024').median().round(1).alias('Tasa de homicidios, mediana'),
    ((pl.col('percentil_homicidios_2022_2024') >= 80).mean() * 100).round(1).alias('% en el 20 % más violento'),
    ((pl.col('quien_ejerce_seguridad').str.starts_with('sin policía')).mean() * 100).round(1).alias('% sin policía municipal'),
).sort('semaforo').rename({'semaforo': 'Semáforo'}))

Municipios con tasa: 2,470 · sin tasa (clave ausente de la estadística algún año o sin población): 8


Semáforo,Municipios,"Tasa de homicidios, mediana",% en el 20 % más violento,% sin policía municipal
str,u32,f64,f64,f64
"""amarillo""",496,10.7,21.4,19.8
"""rojo""",495,9.2,19.1,54.3
"""verde""",1486,9.8,20.0,10.1


**Qué vemos:**

- **El rojo no coincide con lo más violento:** el 19 % de los municipios en rojo está en el 20 % con más homicidios, contra el 20 % de los verdes. El riesgo de abstención y la violencia son cosas distintas, y por eso van en capas separadas.
- **El 54 % de los rojos no tiene policía municipal,** contra el 10 % de los verdes. Son los municipios de usos y costumbres de Oaxaca con policía comunitaria; ya vimos que no tener policía no va con votar menos (notebook 03b, sección 8).
- Un municipio rojo **y** violento sí merece una lectura conjunta: ahí la intervención tiene que considerar la seguridad de quienes la hacen.

## 5 · Vista por personas: dónde está la gente en riesgo

Un municipio chico en rojo pesa lo mismo en el mapa que una ciudad en rojo. Esta vista cuenta **personas**, para que el INE vea dónde se concentra el esfuerzo.

In [8]:
tablero = (semaforo.join(personas, on='cvegeo', how='left').join(riesgo.select('cvegeo', 'estado', 'municipio', 'ayuntamiento_mismo_dia_2027', 'es_sni'), on='cvegeo'))
(tablero.group_by('semaforo').agg(
    pl.len().alias('Municipios'),
    pl.col('lista_2024').sum().alias('Personas en la lista 2024'),
    pl.col('primerizos_votaron_2024').sum().alias('Primerizos que votaron en 2024'),
    pl.col('jovenes_21_29_no_votaron_2024').sum().alias('Jóvenes 21-29 que no votaron'),
    pl.col('nuevos_votantes_2027').sum().alias('Nuevos votantes 2027 (est.)'),
    pl.col('nuevos_sin_credencial_2027_si_se_repite_2021').sum().alias('Sin credencial si se repite 2021 (est.)'),
    (~pl.col('ayuntamiento_mismo_dia_2027')).sum().alias('Sin elección municipal ese día'),
).sort('semaforo').rename({'semaforo': 'Semáforo'}))

Semáforo,Municipios,Personas en la lista 2024,Primerizos que votaron en 2024,Jóvenes 21-29 que no votaron,Nuevos votantes 2027 (est.),Sin credencial si se repite 2021 (est.),Sin elección municipal ese día
str,u32,i64,i64,i64,i64,i64,u32
"""amarillo""",496,41318960,1193288,4136226,1754169,488027,206
"""rojo""",495,24955268,618518,3011632,1157532,332396,404
"""verde""",1486,32056120,1086775,2688108,1451754,448324,66


In [9]:
# Rojo por estado
(tablero.filter(pl.col('semaforo') == 'rojo').group_by('estado').agg(
    pl.len().alias('Municipios en rojo'), pl.col('lista_2024').sum().alias('Personas en rojo'),
    (pl.col('probabilidad_rojo') >= 0.7).sum().alias('Rojos firmes'), pl.col('es_sni').sum().alias('De usos y costumbres'))
 .join(tablero.group_by('estado').agg(lista_estado=pl.col('lista_2024').sum()), on='estado')
 .with_columns((pl.col('Personas en rojo') / pl.col('lista_estado') * 100).round(1).alias('% de la lista del estado')).drop('lista_estado')
 .sort('Personas en rojo', descending=True).rename({'estado': 'Estado'}))

Estado,Municipios en rojo,Personas en rojo,Rojos firmes,De usos y costumbres,% de la lista del estado
str,u32,i64,u32,u32,f64
"""Veracruz de Ignacio de la Llave""",69,3803403,21,0,62.6
"""Baja California""",6,3182648,5,0,99.5
"""Jalisco""",8,1900661,2,0,28.7
"""Sonora""",10,1787560,4,0,78.0
"""Chihuahua""",5,1484646,2,0,48.2
"""Nuevo León""",11,1426494,7,0,31.9
"""Durango""",29,1320624,18,0,94.5
"""Tamaulipas""",3,1210770,2,0,42.5
"""Guanajuato""",8,1180424,0,0,24.4


In [10]:
# Los 15 municipios en rojo con más personas
(tablero.filter(pl.col('semaforo') == 'rojo').sort('lista_2024', descending=True).head(15).select(
    pl.col('estado').alias('Estado'), pl.col('municipio').alias('Municipio'), pl.col('lista_2024').alias('Lista 2024'),
    pl.col('posicion_riesgo').round(1).alias('Posición de riesgo'), (pl.col('probabilidad_rojo') * 100).round(0).alias('Probabilidad de rojo (%)'),
    pl.col('jovenes_21_29_no_votaron_2024').alias('Jóvenes 21-29 que no votaron'), pl.col('primerizos_votaron_2024').alias('Primerizos 2024')))

Estado,Municipio,Lista 2024,Posición de riesgo,Probabilidad de rojo (%),Jóvenes 21-29 que no votaron,Primerizos 2024
str,str,i64,f64,f64,i64,i64
"""Baja California""","""Tijuana""",1644276,92.8,93.0,203705,37843
"""Chihuahua""","""Juárez""",1249327,90.8,87.0,146531,28944
"""Baja California""","""Mexicali""",866896,90.5,85.0,95482,17854
"""Sinaloa""","""Culiacán""",740231,81.4,45.0,82215,19622
"""Quintana Roo""","""Benito Juárez""",699032,82.7,51.0,87588,19099
"""Sonora""","""Hermosillo""",681127,85.2,59.0,85963,17108
"""Guerrero""","""Acapulco de Juárez""",600442,84.0,55.0,72377,13041
"""Tamaulipas""","""Reynosa""",581349,91.2,88.0,72148,14124
"""Tabasco""","""Centro""",529215,81.0,44.0,52410,15750


**Qué vemos:**

- **En número de municipios, el rojo es sobre todo Oaxaca:** 299 de 495, todos de usos y costumbres, con menos de un millón de personas.
- **En número de personas pesan Veracruz (3.8 millones en rojo) y las ciudades grandes.** De las 15 ciudades de la tabla, **13 sí eligen ayuntamiento en 2027**; las excepciones son Durango y Veracruz. Las 13 están en rojo porque votan poco aun con elección local: Tijuana votó 29 % en 2015 y 38 % en 2021.
- **Las dos vistas cuentan historias distintas** y el mapa debe ofrecer las dos. Donde no hay elección local, la intermedia solo trae la elección federal. En las ciudades con elección local, el problema es que se vota poco siempre, sobre todo entre los jóvenes: ahí están 2.3 de los 3.0 millones de jóvenes en rojo que no votaron en 2024.

## 5b · ¿Y si se interviene en los municipios en rojo? (D22)

Las bases del reto piden comparar un escenario **sin** intervención con otro **con** intervención hipotética. No tenemos ninguna medición de cuánto funciona una campaña (D19), así que **no estimamos efectos**: hacemos una **cuenta con tres supuestos explícitos**, cada uno anclado en un patrón que medimos, y solo en los municipios en rojo, donde se concentraría el recurso.

| Supuesto | Qué suponemos | De dónde sale |
|---|---|---|
| **S1 · Retener a los primerizos** | La generación que votó por primera vez en 2024 (18 a 20 años) no cae más que su municipio en la intermedia | En 2021, la generación primeriza de 2018 cayó 6.1 pp más que su municipio (notebook 03b, sección 1). Votos = generación × 6.1 % |
| **S2 · Cerrar la mitad de la brecha joven** | Los jóvenes de 20 a 29 años recortan a la mitad su distancia con el resto del municipio | La brecha de 20 a 29 años en la intermedia de 2021, de cada municipio (típica: −9.9 pp). Votos = jóvenes × mitad de la brecha |
| **S3 · Credencializar a los nuevos** | Los nuevos votantes que quedarían sin credencial la tramitan y votan como los de 18 y 19 años de su municipio en 2021 | Columna `nuevos_sin_credencial_2027_si_se_repite_2021` y participación de 18 y 19 años en 2021. **Suma votos y también personas a la lista** |

**Cómo se calcula:** participación con intervención = (votos previstos + votos adicionales) / (lista 2024 + nuevos en la lista). Para decir si un municipio **sale del rojo** usamos una línea fija: la del semáforo sin intervención. (El semáforo se define como el 20 % de mayor riesgo; si no fijamos la línea, siempre habría 20 % en rojo.)

**Qué esperamos:** efectos chicos frente a la distancia que separa a la mayoría de los rojos del resto, porque la intervención solo toca a una parte de la lista.

In [11]:
EXTRA_PRIMERIZOS = 0.061   # caída extra típica de la generación primeriza, 2018 → 2021 (notebook 03b, sección 1)
GRUPO_JOVEN = ['20-24', '25-29']
lista_2021 = pl.read_parquet(P / 'lista_edad_sexo_municipal.parquet').filter(pl.col('AELEC') == 2021)
patrones_2021 = (lista_2021.group_by('cvegeo').agg(
        p_total=pl.col('SV').sum() / (pl.col('SV') + pl.col('NV')).sum(),
        p_20_29=pl.col('SV').filter(pl.col('grupo_edad').is_in(GRUPO_JOVEN)).sum()
                / (pl.col('SV') + pl.col('NV')).filter(pl.col('grupo_edad').is_in(GRUPO_JOVEN)).sum(),
        participacion_18_19_2021=pl.col('SV').filter(pl.col('grupo_edad') == '18-19').sum()
                                 / (pl.col('SV') + pl.col('NV')).filter(pl.col('grupo_edad') == '18-19').sum())
    .with_columns(pl.col('p_total', 'p_20_29', 'participacion_18_19_2021').fill_nan(None))
    .with_columns(brecha_jovenes_2021=pl.col('p_20_29') - pl.col('p_total')))
print(f'Brecha de 20 a 29 años en 2021, mediana municipal: {patrones_2021["brecha_jovenes_2021"].median() * 100:.1f} pp')

# Votos que sumaría cada supuesto en cada municipio (vacío = sin dato para calcularlo)
supuestos = (personas.select('cvegeo', 'cohorte_18_20_2024', 'jovenes_20_29_2024', 'nuevos_sin_credencial_2027_si_se_repite_2021')
    .join(patrones_2021.select('cvegeo', 'brecha_jovenes_2021', 'participacion_18_19_2021'), on='cvegeo', how='left')
    .with_columns(
        interv_votos_retener_primerizos=(pl.col('cohorte_18_20_2024') * EXTRA_PRIMERIZOS).round(0).cast(pl.Int64),
        interv_votos_brecha_joven=(pl.col('jovenes_20_29_2024') * (-pl.col('brecha_jovenes_2021')).clip(lower_bound=0) / 2).round(0).cast(pl.Int64),
        interv_votos_credencializar=(pl.col('nuevos_sin_credencial_2027_si_se_repite_2021') * pl.col('participacion_18_19_2021')).round(0).cast(pl.Int64),
        interv_nuevos_en_lista=pl.col('nuevos_sin_credencial_2027_si_se_repite_2021')))
COLUMNAS_INTERV = ['interv_votos_retener_primerizos', 'interv_votos_brecha_joven', 'interv_votos_credencializar', 'interv_nuevos_en_lista']

def participacion_con(part, lista, votos, nuevos):
    # part en proporción; votos y nuevos ya en cero donde no aplica
    return (part * lista + votos) / (lista + nuevos)

def lineas_semaforo(d, col):
    # punto medio entre el último rojo y el primero que no lo es (y lo mismo para amarillo/verde)
    rojo_max, resto_min = d.filter(pl.col('semaforo') == 'rojo')[col].max(), d.filter(pl.col('semaforo') != 'rojo')[col].min()
    amar_max, verde_min = d.filter(pl.col('semaforo') == 'amarillo')[col].max(), d.filter(pl.col('semaforo') == 'verde')[col].min()
    assert rojo_max < resto_min and amar_max < verde_min, 'los colores se enciman'
    return (rojo_max + resto_min) / 2, (amar_max + verde_min) / 2

# Nivel nacional estimado (D27), para leer el efecto
base = (semaforo.select('cvegeo', 'semaforo', 'lista_2024').join(riesgo.select('cvegeo', 'estado', 'ayuntamiento_mismo_dia_2027',
        part=pl.col('participacion_prevista_estimado') / 100), on='cvegeo').join(supuestos, on='cvegeo', how='left'))
LINEA_ROJO, _ = lineas_semaforo(base, 'part')
rojo = pl.col('semaforo') == 'rojo'
def cero(c): return pl.when(rojo).then(pl.col(c).fill_null(0)).otherwise(0)
VOTOS = {'S1 · Retener a los primerizos': cero('interv_votos_retener_primerizos'),
         'S2 · Cerrar la mitad de la brecha joven': cero('interv_votos_brecha_joven'),
         'S3 · Credencializar a los nuevos': cero('interv_votos_credencializar')}
filas = []
for nombre, votos in [*VOTOS.items(), ('Los tres juntos (escenario del mapa)', sum(VOTOS.values()))]:
    # D32: si falta el dato de votos de S3, tampoco se suman personas a la lista (escenario de componentes conocidos)
    nuevos = (pl.when(rojo & pl.col('interv_votos_credencializar').is_not_null()).then(pl.col('interv_nuevos_en_lista').fill_null(0)).otherwise(0)
              if nombre.startswith(('S3', 'Los tres')) else pl.lit(0))
    d = base.with_columns(votos=votos, nuevos=nuevos).with_columns(con=participacion_con(pl.col('part'), pl.col('lista_2024'), pl.col('votos'), pl.col('nuevos')))
    rr = d.filter(rojo).with_columns(efecto=(pl.col('con') - pl.col('part')) * 100)
    salen = rr.filter(pl.col('con') > LINEA_ROJO)
    filas.append({'Supuesto': nombre, 'Votos adicionales': int(d['votos'].sum()), 'Efecto en el rojo típico (pp)': round(rr['efecto'].median(), 2),
                  'Rojos que salen del rojo': salen.height, 'Personas en esos municipios': int(salen['lista_2024'].sum()),
                  'Salen, con elección municipal': salen.filter(pl.col('ayuntamiento_mismo_dia_2027')).height,
                  'Salen, sin elección municipal': salen.filter(~pl.col('ayuntamiento_mismo_dia_2027')).height,
                  'Participación nacional (pp)': round(((d['part'] * d['lista_2024']).sum() + d['votos'].sum()) / (d['lista_2024'].sum() + d['nuevos'].sum()) * 100
                                                       - NIVEL_2027, 2)})
print(f'Rojos sin dato para S2 o S3 (no suman): {base.filter(rojo, pl.col("brecha_jovenes_2021").is_null()).height}')
print(f'Distancia típica de un rojo a la línea del rojo: {(LINEA_ROJO - base.filter(rojo)["part"]).median() * 100:.1f} pp')
pl.DataFrame(filas)

Brecha de 20 a 29 años en 2021, mediana municipal: -9.9 pp
Rojos sin dato para S2 o S3 (no suman): 3
Distancia típica de un rojo a la línea del rojo: 5.7 pp


Supuesto,Votos adicionales,Efecto en el rojo típico (pp),Rojos que salen del rojo,Personas en esos municipios,"Salen, con elección municipal","Salen, sin elección municipal",Participación nacional (pp)
str,i64,f64,i64,i64,i64,i64,f64
"""S1 · Retener a los primerizos""",84789,0.32,19,1284966,9,10,0.09
"""S2 · Cerrar la mitad de la brecha joven""",307818,1.28,68,4700814,21,47,0.31
"""S3 · Credencializar a los nuevos""",135793,0.0,7,85229,1,6,-0.03
"""Los tres juntos (escenario del mapa)""",528400,1.59,84,6068544,25,59,0.36


**Qué vemos:** (nivel nacional estimado, 51.4 %)

- **Aun con los tres supuestos juntos, 411 de los 495 rojos siguen en rojo.** La intervención suma 528 mil votos y sube 1.59 pp al municipio rojo típico (de 1.0 a 2.3 pp en 8 de cada 10), pero un rojo típico está 5.7 pp por debajo de la línea.
- **Salen del rojo 84 municipios con 6.1 millones de personas:** 59 sin elección municipal y 25 con elección. Son los que estaban más cerca de la línea.
- **El supuesto que más pesa es cerrar la brecha joven (S2):** 308 mil votos y 68 municipios fuera del rojo. Retener a los primerizos (S1) suma 85 mil votos y saca a 19.
- **Credencializar (S3) suma 136 mil votos, pero no sube el porcentaje** (0.0 pp en el rojo típico; en 188 de los 495 rojos lo baja un poco): las personas nuevas también entran a la lista y votan menos que el promedio de su municipio. **Sirve para incluir a más gente, no para subir la tasa.**
- **A nivel nacional el efecto es chico:** +0.36 pp, porque los rojos tienen una cuarta parte de la lista.
- **Componentes sin dato (D32):** en 73 rojos no hay participación de 18 y 19 años en 2021, así que S3 no se puede calcular; en 6 y 9 faltan S1 y S2 (sin conteo por edad en 2024 o sin brecha de 2021). Ahí se suma solo lo conocido y el escenario se marca `intervencion_parcial`. En Oxchuc y San Mateo del Mar ya no se agregan personas a la lista sin sus votos.
- **Lectura para el INE:** las campañas para jóvenes ayudan, pero no compensan que en 2027 no haya elección municipal en la mayoría de los rojos. Ahí se necesita otra estrategia, que estos datos no pueden medir.

**Advertencias:**

- **Son supuestos, no efectos medidos.** La "mitad" de S2 la elegimos nosotras; S1 aplica la caída típica (6.1 pp) igual en todos los municipios. En el mapa se etiqueta como "supuesto, no efecto medido".
- **El tamaño de los grupos es el de 2024.** Los jóvenes de 20 a 29 años de 2027 no son exactamente las mismas personas.
- **El denominador es la lista de 2024 completa**, con el "no sabemos" incluido, así que el efecto en puntos queda un poco por debajo (más prudente).

## 6 · Archivos para el mapa

Escribimos los tres archivos de `docs/contrato-datos.md` en `data/processed/mapa/resultados-1.6/`:

- **`municipios.csv`:** una fila por municipio, con identidad, historia 2009–2024, semáforo, personas en juego, alertas y contexto.
- **`escenarios_2027.csv`:** una fila por municipio y tipo de intervención, con el nivel nacional estimado (D27): `ninguna`, `retener_primerizos`, `jovenes_20_29`, `credencializacion` y `combinada` (D22, D28). Cinco filas por municipio.
- **`metadatos.json`:** unidades, significado de cada columna, reglas, fuentes y advertencias.

**Qué cambia respecto a `resultados-1.0`:** se agregan el escenario con intervención (`dirigida_rojos`), las columnas `interv_*` con lo que suma cada supuesto en cada municipio y `usos_y_costumbres`, que incluye a los 7 municipios fuera de Oaxaca. Las columnas y los escenarios sin intervención no cambian.

**Qué cambia respecto a `resultados-1.1`:** solo que las celdas sin dato llegan vacías y nunca como `NaN` (eran 15 en `brecha_jovenes_2024`, municipios cuya participación sale de las actas). Lo detectó Jorge al integrar la 1.1.

**Qué cambia respecto a `resultados-1.2`:** en los 15 municipios cuyos conteos de 2024 no tienen a nadie con dato (su participación sale de las actas), las personas en juego por edad y los supuestos S1 y S2 llegan vacíos, no en 0. Los totales no cambian.

**Qué cambia respecto a `resultados-1.3`:** un solo nivel nacional, estimado con el calendario (51.4 %, D27), en lugar de tres; el rango de cada municipio suma el error de ese nivel; y cinco tipos de intervención en lugar de uno (D28). `municipios.csv` no cambia.

**Qué cambia respecto a `resultados-1.4`:** `municipios.csv` agrega cinco columnas, `aporte_historia`, `aporte_calendario`, `aporte_lista`, `aporte_social` y `aporte_politica`: cuántos puntos sube o baja cada grupo de variables la predicción del municipio frente al país (D29, notebook 04c). `metadatos.json` agrega la participación del país en cada elección de 2009 a 2024. `escenarios_2027.csv` no cambia.

**Qué cambia respecto a `resultados-1.5`:** todo el pronóstico se recalcula con el modelo Delta con guardia (D33) y con Zaragoza 2012 corregido (D31): participación, rango, posición, semáforo, probabilidad de rojo, aportes y escenarios. `posicion_riesgo` y `probabilidad_rojo` llevan 4 decimales, para que el corte del semáforo se lea sin ambigüedad. En los municipios en rojo donde falta algún componente de la intervención, el escenario se marca `intervencion_parcial` en `motivo` y, si falta S3, no se agregan personas a la lista (D32). Las columnas no cambian.

In [12]:
# Usos y costumbres fuera de Oaxaca (notebook 03): los que no eligen ayuntamiento por partidos en 2027
sni_fuera = pl.read_csv(raiz / 'data/reference/municipios_sni_fuera_oaxaca.csv', schema_overrides={'cvegeo': pl.String}).filter(pl.col('anio_jornada') == 2027)
SNI_FUERA_2027 = (sni_fuera.join(riesgo.select('cvegeo', 'ayuntamiento_mismo_dia_2027'), on='cvegeo')
                  .filter(~pl.col('ayuntamiento_mismo_dia_2027'))['cvegeo'])
assert SNI_FUERA_2027.len() == 7, SNI_FUERA_2027.to_list()
claves_ine = (pl.read_csv(P / 'municipios_para_mapa.csv', infer_schema=False).filter(pl.col('anio') == '2024')
              .select('cvegeo', clave_ine_2024=pl.col('cve_ent_ine') + pl.col('cve_mun_ine')))
historia = (panel.with_columns((pl.col('participacion') / 100).round(4).alias('part'))
            .pivot(on='AELEC', index='cvegeo', values='part', sort_columns=True)
            .rename({str(a): f'part_{a}' for a in [2009, 2012, 2015, 2018, 2021, 2024]}))
p24 = panel.filter(pl.col('AELEC') == 2024).select('cvegeo', nombre_mpio='municipio', nombre_edo='estado', ln_2024='LN',
                                                   ns_2024=(pl.col('NS') / pl.col('LN')).round(4), territorio_aproximado_2024='territorio_aproximado')
COLUMNAS_PART = [f'part_{a}' for a in [2009, 2012, 2015, 2018, 2021, 2024]]
COLUMNAS_APORTE = ['aporte_historia', 'aporte_calendario', 'aporte_lista', 'aporte_social', 'aporte_politica']

municipios = (p24.join(claves_ine, on='cvegeo', how='left').join(historia, on='cvegeo', how='left')
    .join(semaforo.select('cvegeo', pl.col('posicion_riesgo').round(4), 'semaforo', pl.col('probabilidad_rojo').round(4)), on='cvegeo', how='left')   # 4 decimales: el corte se lee sin ambigüedad (auditoría H007)
    .join(personas.drop('base_2024'), on='cvegeo', how='left')
    .join(indicadores.select('cvegeo', 'alerta_sin_casilla', 'ultima_eleccion_sin_casilla', 'elecciones_por_debajo_del_pais', 'zona_participacion_2024',
                             pl.col('brecha_jovenes_2024').round(2), 'ayuntamiento_mismo_dia_2027', 'gobernador_mismo_dia_2027', 'es_sni',
                             'quien_ejerce_seguridad', 'tiene_policia_municipal', 'policias_municipales_por_10mil_habitantes', 'justicia_civica',
                             'comparacion_homicidios_inegi_sesnsp', pl.col('cifra_oculta_estatal_2025').round(1)), on='cvegeo', how='left')
    .join(violencia.select('cvegeo', 'homicidios_2022_2024', 'tasa_homicidios_2022_2024', 'percentil_homicidios_2022_2024'), on='cvegeo', how='left')
    .join(supuestos.select('cvegeo', *COLUMNAS_INTERV), on='cvegeo', how='left')
    .join(aportes.select('cvegeo', *COLUMNAS_APORTE), on='cvegeo', how='left')
    .with_columns(usos_y_costumbres=pl.col('es_sni') | pl.col('cvegeo').is_in(SNI_FUERA_2027.implode()))
    .with_columns(historia_completa=pl.all_horizontal([pl.col(c).is_not_null() for c in COLUMNAS_PART]),
                  peso_jov_2024=(pl.col('n_jov_2024') / pl.col('ln_2024')).round(4),
                  credencializacion_18_19_2021=(pl.col('credencializacion_18_19_2021') / 100).round(4))
    .sort('cvegeo'))
ORDEN = ['cvegeo', 'clave_ine_2024', 'nombre_mpio', 'nombre_edo', 'ln_2024', *COLUMNAS_PART, 'ns_2024', 'historia_completa', 'territorio_aproximado_2024',
         'posicion_riesgo', 'semaforo', 'probabilidad_rojo', *COLUMNAS_APORTE,
         'peso_jov_2024', 'n_jov_2024', 'nv_jov_2024', 'primerizos_votaron_2024', 'jovenes_21_29_lista_2024', 'jovenes_21_29_no_votaron_2024',
         'nuevos_votantes_2027', 'credencializacion_18_19_2021', 'nuevos_sin_credencial_2027_si_se_repite_2021', *COLUMNAS_INTERV,
         'alerta_sin_casilla', 'ultima_eleccion_sin_casilla', 'elecciones_por_debajo_del_pais', 'zona_participacion_2024', 'brecha_jovenes_2024',
         'ayuntamiento_mismo_dia_2027', 'gobernador_mismo_dia_2027', 'es_sni', 'usos_y_costumbres',
         'homicidios_2022_2024', 'tasa_homicidios_2022_2024', 'percentil_homicidios_2022_2024', 'comparacion_homicidios_inegi_sesnsp', 'cifra_oculta_estatal_2025',
         'quien_ejerce_seguridad', 'tiene_policia_municipal', 'policias_municipales_por_10mil_habitantes', 'justicia_civica']
municipios = municipios.select(ORDEN)

filas = []
for nivel, valor in ESCENARIOS.items():
    filas.append(riesgo.select(
        'cvegeo', anio=pl.lit(2027), nivel_nacional=pl.lit(nivel), intervencion=pl.lit('ninguna'), escenario_id=pl.lit(f'nac-{nivel}_int-ninguna'),
        part_pred=(pl.col(f'participacion_prevista_{nivel}') / 100), nivel=pl.lit(valor / 100)))
Z90 = 1.645   # el error del nivel nacional se suma en cuadratura al del municipio (errores independientes)
escenarios = (pl.concat(filas)
    .join(semaforo.select('cvegeo', 'q05', 'q95', 'semaforo'), on='cvegeo', how='left')
    .with_columns(
        desv_base=(pl.col('part_pred') - pl.col('nivel')).round(4) + 0.0,
        efecto_interv=pl.lit(0.0),
        part_pred_inf=(pl.col('part_pred') - (pl.col('q05') ** 2 + (Z90 * ERROR_NACIONAL) ** 2).sqrt() / 100).clip(0, 1).round(4),
        part_pred_sup=(pl.col('part_pred') + (pl.col('q95') ** 2 + (Z90 * ERROR_NACIONAL) ** 2).sqrt() / 100).clip(0, 1).round(4),
        part_pred=pl.col('part_pred').round(4),
        estado_estimacion=pl.when(pl.col('part_pred').is_not_null()).then(pl.lit('estimado')).otherwise(pl.lit('sin_estimacion')),
        riesgo=pl.col('semaforo').replace_strict({'rojo': 'alto', 'amarillo': 'medio', 'verde': 'bajo'}, default='sin_estimacion'),
        votos_adicionales=pl.lit(None, dtype=pl.Int64))
    .join(municipios.select('cvegeo', 'territorio_aproximado_2024', 'alerta_sin_casilla'), on='cvegeo', how='left')
    .with_columns(motivo=pl.concat_str([
        pl.when(pl.col('territorio_aproximado_2024')).then(pl.lit('territorio_aproximado')),
        pl.when(pl.col('alerta_sin_casilla') != 'sin antecedente').then(pl.lit('antecedente_sin_casilla'))], separator='|', ignore_nulls=True))
    .with_columns(motivo=pl.when(pl.col('motivo') == '').then(None).otherwise(pl.col('motivo')))
    .select('cvegeo', 'anio', 'nivel_nacional', 'intervencion', 'escenario_id', 'desv_base', 'efecto_interv', 'part_pred', 'part_pred_inf', 'part_pred_sup',
            'estado_estimacion', 'motivo', 'riesgo', 'votos_adicionales')
    .sort('cvegeo', 'nivel_nacional'))

# Escenarios con intervención (D22, D28): cada tipo por separado y los tres juntos, solo en los municipios en rojo
TIPOS = {'retener_primerizos': (['interv_votos_retener_primerizos'], False),
         'jovenes_20_29': (['interv_votos_brecha_joven'], False),
         'credencializacion': (['interv_votos_credencializar'], True),
         'combinada': (['interv_votos_retener_primerizos', 'interv_votos_brecha_joven', 'interv_votos_credencializar'], True)}
con = []
for (nivel, (INTERVENCION, (cols_votos, suma_lista))) in [(n, t) for n in ESCENARIOS for t in TIPOS.items()]:
    e = (escenarios.filter(pl.col('nivel_nacional') == nivel).join(municipios.select('cvegeo', 'ln_2024', 'semaforo', *COLUMNAS_INTERV), on='cvegeo')
         .join(riesgo.select('cvegeo', part_bruta=pl.col(f'participacion_prevista_{nivel}') / 100), on='cvegeo'))   # sin redondear, para no empatar en la línea
    linea_rojo, linea_amarillo = lineas_semaforo(e, 'part_bruta')
    aplica = pl.col('riesgo') == 'alto'
    e = (e.with_columns(
            votos=pl.when(aplica).then(pl.sum_horizontal(pl.col(*cols_votos).fill_null(0))).otherwise(0),
            nuevos=pl.when(aplica & pl.lit(suma_lista) & pl.col('interv_votos_credencializar').is_not_null()).then(pl.col('interv_nuevos_en_lista').fill_null(0)).otherwise(0),
            # D32: algún componente del tipo no tiene dato; se suma solo lo conocido y se marca
            parcial=aplica & pl.any_horizontal(pl.col(*cols_votos).is_null()))
         .with_columns(con=pl.when(aplica).then(participacion_con(pl.col('part_bruta'), pl.col('ln_2024'), pl.col('votos'), pl.col('nuevos')))
                       .otherwise(pl.col('part_bruta')))
         .with_columns(efecto_interv=pl.when(aplica).then(pl.col('con') - pl.col('part_bruta')).otherwise(0.0).round(4) + 0.0)   # + 0.0 evita escribir -0.0
         .with_columns(
            intervencion=pl.lit(INTERVENCION), escenario_id=pl.lit(f'nac-{nivel}_int-{INTERVENCION}'),
            desv_base=(pl.col('desv_base') + pl.col('efecto_interv')).round(4) + 0.0,
            part_pred_inf=(pl.col('part_pred_inf') + pl.col('efecto_interv')).clip(0, 1).round(4),
            part_pred_sup=(pl.col('part_pred_sup') + pl.col('efecto_interv')).clip(0, 1).round(4),
            part_pred=pl.col('con').round(4),
            riesgo=pl.when(pl.col('con') <= linea_rojo).then(pl.lit('alto')).when(pl.col('con') <= linea_amarillo).then(pl.lit('medio')).otherwise(pl.lit('bajo')),
            votos_adicionales=pl.col('votos').cast(pl.Int64))
         .with_columns(motivo=pl.when(pl.col('parcial')).then(pl.concat_str([pl.col('motivo'), pl.lit('intervencion_parcial')], separator='|', ignore_nulls=True))
                       .otherwise(pl.col('motivo'))))
    con.append(e.select(escenarios.columns))
escenarios = pl.concat([escenarios, *con]).sort('cvegeo', 'intervencion', 'nivel_nacional')
# -0.0 y 0.0 son el mismo número; se escribe siempre 0.0
escenarios = escenarios.with_columns([pl.when(pl.col(c) == 0).then(0.0).otherwise(pl.col(c)).alias(c) for c, t in escenarios.schema.items() if t == pl.Float64])
print(escenarios.group_by('intervencion', 'nivel_nacional', 'riesgo').len().pivot(on='riesgo', index=['intervencion', 'nivel_nacional'], values='len').sort('intervencion', 'nivel_nacional'))
print(f'municipios.csv: {municipios.height:,} filas × {municipios.width} columnas · escenarios_2027.csv: {escenarios.height:,} filas')

shape: (5, 5)
┌────────────────────┬────────────────┬──────┬──────┬───────┐
│ intervencion       ┆ nivel_nacional ┆ bajo ┆ alto ┆ medio │
│ ---                ┆ ---            ┆ ---  ┆ ---  ┆ ---   │
│ str                ┆ str            ┆ u32  ┆ u32  ┆ u32   │
╞════════════════════╪════════════════╪══════╪══════╪═══════╡
│ combinada          ┆ estimado       ┆ 1486 ┆ 411  ┆ 580   │
│ credencializacion  ┆ estimado       ┆ 1486 ┆ 488  ┆ 503   │
│ jovenes_20_29      ┆ estimado       ┆ 1486 ┆ 427  ┆ 564   │
│ ninguna            ┆ estimado       ┆ 1486 ┆ 495  ┆ 496   │
│ retener_primerizos ┆ estimado       ┆ 1486 ┆ 476  ┆ 515   │
└────────────────────┴────────────────┴──────┴──────┴───────┘
municipios.csv: 2,477 filas × 53 columnas · escenarios_2027.csv: 12,385 filas


In [13]:
# Comprobaciones del contrato antes de escribir
# Una celda sin dato va vacía, nunca como NaN (contrato)
municipios = municipios.with_columns([pl.col(c).fill_nan(None) for c, t in municipios.schema.items() if t.is_float()])
escenarios = escenarios.with_columns([pl.col(c).fill_nan(None) for c, t in escenarios.schema.items() if t.is_float()])
for t in (municipios, escenarios):
    assert not any(t[c].is_nan().any() for c, tipo in t.schema.items() if tipo.is_float()), 'hay NaN'
assert municipios['cvegeo'].n_unique() == municipios.height and municipios['cvegeo'].str.len_chars().eq(5).all()
assert set(escenarios['cvegeo']) == set(municipios['cvegeo'])
assert escenarios.group_by('cvegeo').len()['len'].eq(len(ESCENARIOS) * (1 + len(TIPOS))).all()
assert municipios['usos_y_costumbres'].sum() == municipios['es_sni'].sum() + 7
for c in COLUMNAS_PART + ['part_pred', 'part_pred_inf', 'part_pred_sup', 'ns_2024', 'probabilidad_rojo', 'peso_jov_2024']:
    t = municipios if c in municipios.columns else escenarios
    assert t[c].drop_nulls().is_between(0, 1).all(), c
assert (escenarios['part_pred_inf'] <= escenarios['part_pred']).all() and (escenarios['part_pred'] <= escenarios['part_pred_sup']).all()
# El promedio ponderado por la lista de cada escenario debe ser su nivel nacional
for nivel, valor in ESCENARIOS.items():
    e = escenarios.filter(pl.col('nivel_nacional') == nivel, pl.col('intervencion') == 'ninguna').join(municipios.select('cvegeo', 'ln_2024'), on='cvegeo')
    promedio = (e['part_pred'] * e['ln_2024']).sum() / e['ln_2024'].sum()
    assert abs(promedio - valor / 100) < 0.001, (nivel, promedio)
# Sin intervención, el riesgo es el semáforo
sin = escenarios.filter(pl.col('intervencion') == 'ninguna')
assert sin.group_by('cvegeo').agg(pl.col('riesgo').n_unique())['riesgo'].eq(1).all()
# Con intervención: solo cambian los rojos, y nunca baja la participación más que el redondeo... salvo S3, que puede bajarla un poco
c = escenarios.filter(pl.col('intervencion') != 'ninguna').join(sin.select('cvegeo', 'nivel_nacional', r0='riesgo'), on=['cvegeo', 'nivel_nacional'])
assert c.filter(pl.col('r0') != 'alto')['efecto_interv'].eq(0).all() and c.filter(pl.col('r0') != 'alto')['riesgo'].equals(c.filter(pl.col('r0') != 'alto')['r0'])
assert c.filter(pl.col('r0') != 'alto')['votos_adicionales'].eq(0).all()
# Los aportes (D29) suman los puntos frente al país de la predicción sin intervención (salvo redondeo a 4 decimales de part_pred)
chk = sin.join(municipios.select('cvegeo', *COLUMNAS_APORTE), on='cvegeo').filter(pl.col('part_pred').is_not_null())
assert chk.select(COLUMNAS_APORTE).null_count().sum_horizontal().item() == 0
dif = (chk.select(pl.sum_horizontal(COLUMNAS_APORTE))[:, 0] - (chk['part_pred'] - NIVEL_2027 / 100) * 100).abs()
assert dif.max() < 0.05, dif.max()
print(f'Aportes: suman los puntos frente al país (diferencia máxima {dif.max():.3f} pp, por redondeo)')
print('Contrato: todo en orden.')
print(escenarios.group_by('intervencion', 'nivel_nacional', 'riesgo').len().pivot(on='riesgo', index=['intervencion', 'nivel_nacional'], values='len').sort('intervencion', 'nivel_nacional'))

Aportes: suman los puntos frente al país (diferencia máxima 0.020 pp, por redondeo)
Contrato: todo en orden.
shape: (5, 5)
┌────────────────────┬────────────────┬──────┬──────┬───────┐
│ intervencion       ┆ nivel_nacional ┆ alto ┆ bajo ┆ medio │
│ ---                ┆ ---            ┆ ---  ┆ ---  ┆ ---   │
│ str                ┆ str            ┆ u32  ┆ u32  ┆ u32   │
╞════════════════════╪════════════════╪══════╪══════╪═══════╡
│ combinada          ┆ estimado       ┆ 411  ┆ 1486 ┆ 580   │
│ credencializacion  ┆ estimado       ┆ 488  ┆ 1486 ┆ 503   │
│ jovenes_20_29      ┆ estimado       ┆ 427  ┆ 1486 ┆ 564   │
│ ninguna            ┆ estimado       ┆ 495  ┆ 1486 ┆ 496   │
│ retener_primerizos ┆ estimado       ┆ 476  ┆ 1486 ┆ 515   │
└────────────────────┴────────────────┴──────┴──────┴───────┘


In [14]:
VERSION = 'resultados-1.6'
PARTICIPACION_PAIS = dict(panel.group_by('AELEC').agg(pl.col('participacion_nacional').first()).sort('AELEC').select('AELEC', (pl.col('participacion_nacional') / 100).round(4)).iter_rows())
assert len(PARTICIPACION_PAIS) == 6
salida = P / 'mapa' / VERSION
salida.mkdir(parents=True, exist_ok=True)
municipios.write_csv(salida / 'municipios.csv')
escenarios.write_csv(salida / 'escenarios_2027.csv')

unidades = {
    'part_*': 'proporción de 0 a 1 sobre votaron + no votaron (sin NS), 4 decimales; territorio de 2024',
    'ln_2024': 'personas en la lista nominal de 2024',
    'ns_2024': 'proporción de la lista nominal de 2024 con dato "no sabemos"',
    'historia_completa': 'true si hay participación en las seis elecciones',
    'territorio_aproximado_2024': 'true si el territorio del municipio no se pudo reconstruir completo en algún año (D13)',
    'posicion_riesgo': 'de 0 a 100; 100 = el municipio que se prevé más por debajo del país en 2027',
    'semaforo': 'rojo (posición ≥ 80), amarillo (60 a < 80), verde (< 60) (D18)',
    'probabilidad_rojo': 'proporción de 2,000 simulaciones en que el municipio queda en rojo, sumando errores de la validación del modelo',
    'aporte_*': 'puntos porcentuales (2 decimales): cuánto sube (+) o baja (−) cada grupo de variables la predicción del municipio frente al país (D29, notebook 04c). Los cinco suman (part_pred sin intervención − nivel nacional) × 100. Grupos: historia de participación, calendario y usos y costumbres, lista y población, condiciones sociales, competencia política. Es un reparto de la predicción, no causas probadas',
    'peso_jov_2024': 'proporción de la lista 2024 de 18 a 34 años que votó o no votó (mismo concepto que la muestra 0.3)',
    'n_jov_2024': 'personas de 18 a 34 años que votaron o no votaron en 2024',
    'nv_jov_2024': 'personas de 18 a 34 años que no votaron en 2024',
    'primerizos_votaron_2024': 'personas de 18 a 20 años que votaron en 2024 (su primera elección federal)',
    'jovenes_21_29_lista_2024': 'personas de 21 a 29 años en la lista nominal de 2024',
    'jovenes_21_29_no_votaron_2024': 'personas de 21 a 29 años que no votaron en 2024',
    'nuevos_votantes_2027': 'personas que tendrán 18 o 19 años el 6 de junio de 2027, estimadas con el censo 2020 (0.25 × edad 10 + edad 11 + 0.75 × edad 12)',
    'credencializacion_18_19_2021': 'proporción de la generación de 18 y 19 años en la lista nominal de 2021 (puede pasar de 1)',
    'nuevos_sin_credencial_2027_si_se_repite_2021': 'personas; escenario aritmético: nuevos_votantes_2027 × (1 − credencialización 2021, tope 1)',
    'alerta_sin_casilla': 'recurrente (2 o más elecciones con parte de la lista sin casilla), antecedente (1) o sin antecedente (D12); va aparte del modelo',
    'ultima_eleccion_sin_casilla': 'año',
    'elecciones_por_debajo_del_pais': 'número de elecciones de 2009 a 2024 con participación menor a la nacional',
    'zona_participacion_2024': 'agrupamiento territorial de la participación 2024 (LISA, vecindad Queen)',
    'brecha_jovenes_2024': 'puntos porcentuales: participación de 20 a 29 años menos la del municipio, 2024',
    'ayuntamiento_mismo_dia_2027': 'true si el municipio elige ayuntamiento el 6 de junio de 2027 (calendario a octubre de 2026)',
    'gobernador_mismo_dia_2027': 'true si el estado elige gubernatura ese día',
    'es_sni': 'true si es municipio de Oaxaca que elige autoridades por sistemas normativos indígenas (IEEPCO 2025)',
    'usos_y_costumbres': 'true si el municipio elige autoridades por sistemas normativos indígenas: los de es_sni más 7 fuera de Oaxaca que en 2027 no eligen ayuntamiento por partidos (data/reference/municipios_sni_fuera_oaxaca.csv)',
    'interv_votos_retener_primerizos': 'votos; supuesto S1 (D22): generación de 18 a 20 años de 2024 que votó o no votó × 6.1 % (caída extra típica de la generación primeriza 2018 → 2021)',
    'interv_votos_brecha_joven': 'votos; supuesto S2 (D22): personas de 20 a 29 años de 2024 que votaron o no votaron × la mitad de su brecha con el municipio en 2021 (0 si no había brecha); vacío sin dato de 2021',
    'interv_votos_credencializar': 'votos; supuesto S3 (D22): nuevos_sin_credencial_2027_si_se_repite_2021 × participación de 18 y 19 años del municipio en 2021; vacío sin dato',
    'interv_nuevos_en_lista': 'personas que S3 suma a la lista (igual a nuevos_sin_credencial_2027_si_se_repite_2021); vacío sin dato',
    'homicidios_2022_2024': 'defunciones por homicidio, INEGI, por municipio de ocurrencia; vacío si la clave falta algún año',
    'tasa_homicidios_2022_2024': 'homicidios por 100 mil habitantes al año (población CONAPO 2023)',
    'percentil_homicidios_2022_2024': 'de 0 a 100; 100 = la tasa más alta',
    'comparacion_homicidios_inegi_sesnsp': 'etiqueta del estado (D17): qué fuente cuenta más homicidios de 2019 a 2024',
    'cifra_oculta_estatal_2025': 'porcentaje de delitos no denunciados o sin carpeta en el estado, ENVIPE 2026',
    'quien_ejerce_seguridad': 'categoría del INEGI (CNGMD 2025, situación al 31 de diciembre de 2024)',
    'tiene_policia_municipal': 'true / false; vacío si no aplica o no identificado',
    'policias_municipales_por_10mil_habitantes': 'policías municipales (incluye titular) por 10 mil habitantes (CONAPO 2024)',
    'justicia_civica': 'categoría del INEGI (CNGMD 2025)',
    'desv_base': 'proporción; participación prevista menos el nivel nacional del escenario',
    'efecto_interv': 'proporción; participación con intervención menos sin intervención. 0 en intervencion = "ninguna" y fuera del rojo',
    'part_pred': 'proporción de 0 a 1; nivel nacional + desviación prevista, ajustada para que el promedio ponderado por la lista sea el nivel',
    'part_pred_inf, part_pred_sup': 'proporción; intervalo de 90 %: errores de la validación por tamaño de municipio sumados en cuadratura con el error del nivel nacional (D27)',
    'riesgo': 'alto = rojo, medio = amarillo, bajo = verde. Con intervención se compara contra la línea fija del semáforo sin intervención',
    'votos_adicionales': 'votos que suma el tipo de intervención en los municipios en rojo; 0 fuera del rojo; vacío sin intervención',
}
metadatos = {
    'version_resultados': VERSION,
    'generado': datetime.now().isoformat(timespec='seconds'),
    'tipo': 'resultados del análisis (reemplaza a resultados-1.5, 1.4, 1.3, 1.2, 1.1, 1.0 y muestra-0.3)',
    'pregunta': '¿Qué zonas podrían presentar mayor abstención en el futuro si no se interviene? Intermedia del 6 de junio de 2027.',
    'clave': 'cvegeo (INEGI, 5 caracteres con ceros) es la clave estable y de unión con la geometría. clave_ine_2024 es solo referencia.',
    'lectura': 'Leer cvegeo, clave_ine_2024 y motivo como texto. Celda vacía = sin dato o no aplica; nunca equivale a cero.',
    'sin_conteo_2024': 'En 15 municipios los conteos de 2024 no tienen a nadie con dato (su participación sale de las actas): n_jov_2024, nv_jov_2024, peso_jov_2024, primerizos_votaron_2024, jovenes_21_29_no_votaron_2024, brecha_jovenes_2024, interv_votos_retener_primerizos e interv_votos_brecha_joven van vacíos.',
    'archivos': ['municipios.csv', 'escenarios_2027.csv', 'metadatos.json'],
    'niveles_nacionales_2027': {k: round(v / 100, 4) for k, v in ESCENARIOS.items()},
    'modelo_nacional': {
        'decision': 'D27: una sola predicción por municipio; el nivel nacional se estima con el calendario local',
        'recta': f'participación nacional (%) = {ORDENADA:.4f} + {PENDIENTE:.4f} × % de la lista que elige ayuntamiento el mismo día',
        'datos': 'intermedias 2009, 2015 y 2021 de los CCPC: ' + ', '.join(f"{a}: {p} % con {s:.1f} % de la lista con ayuntamiento" for a, p, s in zip(calendario['AELEC'], calendario['participacion_nacional'], calendario['pct_lista_ayuntamiento'])),
        'pct_lista_ayuntamiento_2027': round(PCT_AYTO_2027, 1),
        'nivel_2027': NIVEL_2027,
        'prueba': 'cada intermedia predicha con las otras dos; errores (pp): ' + ', '.join(f"{a}: {e:+.1f}" for a, e in zip(prueba_nacional['Intermedia'], prueba_nacional['Error (pp)'])),
        'error_nacional_pp': round(ERROR_NACIONAL, 2),
        'sensibilidad': 'el semáforo no depende del nivel; con otro nivel N, la participación de cada municipio se mueve casi lo mismo que N − nivel_2027. Niveles de referencia: ' + ', '.join(f'{k} {v} %' for k, v in SENSIBILIDAD.items()),
    },
    'participacion_nacional_historica': {
        'valores': {str(a): v for a, v in PARTICIPACION_PAIS.items()},
        'unidad': 'proporción de 0 a 1 sobre votaron + no votaron, con las mismas correcciones que la historia de cada municipio (notebook 03)',
        'nota': 'El modelo nacional (D27) usa las intermedias de los CCPC sin esas correcciones (44.06, 47.07 y 51.84 %); la diferencia es de 0.03 puntos o menos',
    },
    'aportes': 'D29: aporte_* reparte la predicción de cada municipio entre cinco grupos de variables (Huber exacto y random forest por contribuciones de camino, promediados), centrados en el promedio del país ponderado por la lista de 2024. Con Delta con guardia (D33), la desviación de 2024 de la que parte el modelo cuenta en aporte_historia. Notebook 04c',
    'formula': 'part_pred = nivel nacional estimado + desviación prevista + ajuste; el ajuste hace que el promedio ponderado por ln_2024 sea el nivel',
    'semaforo': {
        'regla': 'posición de riesgo = 100 − percentil de la desviación prevista; rojo si ≥ 80, amarillo si ≥ 60, verde en otro caso (D18)',
        'umbral_rojo_desviacion_pp': round(UMBRAL_ROJO, 4), 'umbral_amarillo_desviacion_pp': round(UMBRAL_AMARILLO, 4),
        'equivalencia_contrato': {'rojo': 'alto', 'amarillo': 'medio', 'verde': 'bajo'},
        'validacion': 'aprendiendo de 2015 y prediciendo 2021, el 70 % de los municipios del 20 % de mayor riesgo previsto quedaron en el 20 % real (regla simple: 67 %)',
    },
    'intervalo': 'errores de la validación (las dos direcciones), centrados, cuantiles 5 y 95 por tamaño de lista (<5 mil, 5-20 mil, 20-100 mil, 100 mil+), sumados en cuadratura con 1.645 × el error del nivel nacional. Es una aproximación condicional: sin la parte nacional, probado de una elección a otra cubrió 93 % (2021) y 79 % (2015), y menos en los municipios de usos y costumbres (88 % y 74 %)',
    'intervencion': {
        'ninguna': 'pronóstico sin intervención',
        'retener_primerizos': 'S1, supuesto, no efecto medido: la generación de 18 a 20 años de 2024 no cae más que su municipio (columna interv_votos_retener_primerizos)',
        'jovenes_20_29': 'S2, supuesto, no efecto medido: los de 20 a 29 años recortan a la mitad su brecha con el municipio (interv_votos_brecha_joven)',
        'credencializacion': 'S3, supuesto, no efecto medido: los nuevos sin credencial la tramitan y votan como los de 18 y 19 años de su municipio en 2021; también suman personas a la lista (interv_votos_credencializar, interv_nuevos_en_lista)',
        'combinada': 'S1 + S2 + S3 juntos',
        'alcance': 'solo en los municipios en rojo; fuera del rojo no cambia nada. No se estima el efecto de una campaña (D19, D22, D28)',
        'comparacion': 'comparar nac-estimado_int-ninguna con nac-estimado_int-<tipo>',
        'formula': 'part_pred con = (part_pred sin × ln_2024 + votos_adicionales) / (ln_2024 + interv_nuevos_en_lista)',
    },
    'escenarios': sorted(escenarios['escenario_id'].unique().to_list()),
    'motivos': {'territorio_aproximado': 'territorio reconstruido de forma aproximada en algún año (D13)',
                'antecedente_sin_casilla': 'parte de su lista se quedó sin casilla en alguna elección de 2009 a 2024 (D12): revisar aparte del semáforo',
                'intervencion_parcial': 'municipio en rojo donde algún componente del tipo de intervención no tiene dato (S1 y S2 sin conteo de 2024; S3 sin participación de 18 y 19 años en 2021): se suma solo lo conocido y, si falta S3, tampoco se agregan personas a la lista (D32)'},
    'motivo_formato': "uno o varios códigos separados por '|'",
    'unidades': unidades,
    'fuentes': ['INE, Conteos Censales de Participación Ciudadana 2009-2024', 'INE, cómputos distritales por casilla 2009-2024',
                'INEGI: Censos 2010 y 2020, Encuesta Intercensal 2015, Estadísticas de Defunciones Registradas, CNGMD 2025, ENVIPE 2026',
                'CONAPO: proyecciones de población base 2020, marginación', 'CONEVAL: pobreza municipal y rezago social', 'SESNSP: incidencia delictiva',
                'IEEPCO: municipios por sistemas normativos indígenas 2025', 'Organismos públicos locales: calendario de elecciones 2027'],
    'advertencias': [
        f'El nivel nacional de 2027 ({NIVEL_2027:.1f} %) es una estimación por calendario hecha con solo tres intermedias; su error de prueba es de unos {ERROR_NACIONAL:.0f} puntos. El semáforo no depende de él.',
        'El intervalo de cada municipio es ancho y aproximado: su error propio va de unos ±7 a ±15 pp según su tamaño y, con el error del nivel nacional, el rango mide de 18 a 29 puntos. No garantiza 90 % en cada grupo ni en 2027. La posición frente a los demás es más confiable que el nivel exacto.',
        'La probabilidad de rojo es la frecuencia en simulaciones con los errores del pasado, no una probabilidad garantizada; es optimista en los extremos: 80 % o más se lee como "probable", no como "seguro".',
        'El calendario local de 2027 puede cambiar; elegir ayuntamiento el mismo día es la variable que más pesa.',
        'La tasa de homicidios de 2022 a 2024 (en logaritmo) es una de las 34 variables del modelo, en el grupo de condiciones sociales, y casi no pesa. Las demás columnas de seguridad (policía, justicia cívica, cifra oculta y comparación de fuentes) son solo contexto: no entran al modelo.',
        'El escenario con intervención es una cuenta con supuestos explícitos, no un efecto medido de una campaña. Donde falta algún componente se suma solo lo conocido y se marca intervencion_parcial (D32).',
        'La validación es retrospectiva con datos revisados: algunas fuentes (por ejemplo, CONEVAL 2020) se publicaron después de la elección que se predice.',
        'No se nombran personas ni se atribuyen causas individuales a ningún municipio.',
    ],
    'modelo': 'Delta con guardia (D33): desviación de la presidencial anterior + cambio que aprende un ensamble (promedio de Huber y random forest, D21) con 34 variables; si en esa presidencial faltaron casillas o no hay dato, ensamble de nivel. Evaluación retrospectiva con datos revisados: aprende de una transición presidencial-intermedia y predice la otra',
    'notebooks': ['03_historia_y_variables', '03b_variables_externas_y_patrones', '04_modelo_riesgo_2027', '04b_fuentes_adicionales_e_importancia', '04c_por_que_cada_municipio', '05_semaforo_y_mapa'],
}
(salida / 'metadatos.json').write_text(json.dumps(metadatos, ensure_ascii=False, indent=2), encoding='utf-8')
faltan_unidades = [c for c in municipios.columns + escenarios.columns
                   if c not in unidades and not c.startswith(('part_', 'aporte_')) and c not in ('cvegeo', 'clave_ine_2024', 'nombre_mpio', 'nombre_edo', 'anio',
                                                                                  'nivel_nacional', 'intervencion', 'escenario_id', 'estado_estimacion', 'motivo')]
assert not faltan_unidades, faltan_unidades
print(f'Archivos escritos en {salida.relative_to(raiz)}:', sorted(p.name for p in salida.iterdir()))

Archivos escritos en data/processed/mapa/resultados-1.6: ['escenarios_2027.csv', 'metadatos.json', 'municipios.csv']


## 7 · Advertencias y siguiente paso

**Advertencias:**

- **El semáforo ordena municipios.** El nivel nacional de 2027 (51.4 %) es una estimación por calendario hecha con tres intermedias, con un error típico de unos 3 puntos; con los niveles de 2009 o 2015, todos los porcentajes bajarían casi lo mismo y el semáforo no cambiaría.
- **El calendario local de 2027 puede cambiar** y es la variable que más pesa. Si un estado mueve o agrega elecciones, hay que volver a correr los notebooks 03b, 04 y 05.
- **Las probabilidades suponen que los errores de los municipios son independientes.** En la realidad se agrupan en regiones (notebook 03b, sección 3), así que pueden fallar varios vecinos a la vez.
- **Las personas en juego son conteos, no efectos.** El escenario con intervención (D22) es una cuenta con supuestos explícitos: dice cuánto sumaría **si** pasara lo que supone, no cuántos votos traería una campaña real.
- **La alerta "sin casilla"** (D12) va aparte del semáforo y siempre se revisa, sin importar el color.

**Siguiente paso. Paso 6, mapa y prueba:** cargar `resultados-1.6` en la aplicación, revisar las vistas (semáforo, personas, contexto de seguridad) y probar que el mapa diga lo mismo que este notebook.